In [1]:
from pathlib import Path
import gzip
import re
import subprocess

import pandas as pd

In [2]:
# Input GTF from Group 1
GTF = Path(
    "Data/group1/Ensembl_reference_transcripts.gtf.gz"
)

# Prepared query GTF
PREPARED_GTF = Path(
    "Data/group1/Ensembl_reference_transcripts.sorted.gtf"
)

# Provenance created when the official index was downloaded
INDEX_PROVENANCE = Path(
    "Output_files/official_57_sample/index_download_provenance.tsv"
)

# Outputs
OUT_DIR = Path(
    "Output_files/official_57_sample"
)

ANNOTATED_OUTPUT = OUT_DIR / "annotated.gtf.gz"

# Isopedia executable
ISOPEDIA = "Data/isopedia"

THREADS = 16

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
def open_text(path):

    if path.suffix == ".gz":
        return gzip.open(
            path,
            "rt",
            encoding="utf-8"
        )

    return open(
        path,
        "r",
        encoding="utf-8"
    )

In [4]:
def get_transcript_id(attributes):

    match = re.search(
        r'transcript_id\s+"([^"]+)"',
        attributes
    )

    if match:
        return match.group(1)

    return None

In [5]:
def prepare_gtf(
    source,
    destination
):

    comments = []

    transcripts = {}
    exons = {}

    with open_text(source) as f:

        for line in f:

            if line.startswith("#"):

                comments.append(line)
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            if fields[2] not in {
                "transcript",
                "exon"
            }:
                continue

            tid = get_transcript_id(
                fields[8]
            )

            if tid is None:
                continue

            if fields[2] == "transcript":

                transcripts[tid] = fields

            else:

                exons.setdefault(
                    tid,
                    []
                ).append(fields)

    # Keep only transcripts that have exons
    valid_ids = [
        tid
        for tid in transcripts
        if tid in exons
    ]

    # Sort transcript models
    valid_ids = sorted(
        valid_ids,
        key=lambda tid: (
            transcripts[tid][0],
            int(transcripts[tid][3]),
            int(transcripts[tid][4]),
            tid
        )
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        destination,
        "w"
    ) as out:

        out.writelines(comments)

        for tid in valid_ids:

            # Transcript row
            out.write(
                "\t".join(
                    transcripts[tid]
                )
                + "\n"
            )

            # Exons
            sorted_exons = sorted(
                exons[tid],
                key=lambda row: (
                    int(row[3]),
                    int(row[4])
                )
            )

            for exon in sorted_exons:

                out.write(
                    "\t".join(exon)
                    + "\n"
                )

    print(
        "Prepared transcripts:",
        len(valid_ids)
    )

    return len(valid_ids)

In [ ]:
query_transcript_count = prepare_gtf(
    GTF,
    PREPARED_GTF
)

print(
    "Prepared GTF:",
    PREPARED_GTF
)

print(
    "Number of transcripts in prepared GTF:",
    query_transcript_count
)

In [6]:
def inspect_gtf(path):

    transcripts = set()

    exon_counts = {}

    rows = 0

    with open_text(path) as f:

        for line in f:

            if (
                not line.strip()
                or line.startswith("#")
            ):
                continue

            fields = (
                line.rstrip("\n")
                .split("\t")
            )

            if len(fields) != 9:
                continue

            rows += 1

            tid = get_transcript_id(
                fields[8]
            )

            if not tid:
                continue

            if fields[2] == "transcript":

                transcripts.add(tid)

            elif fields[2] == "exon":

                exon_counts[tid] = (
                    exon_counts.get(
                        tid,
                        0
                    )
                    + 1
                )

    missing_exons = [
        tid
        for tid in transcripts
        if tid not in exon_counts
    ]

    print(
        "GTF rows:",
        rows
    )

    print(
        "Transcripts:",
        len(transcripts)
    )

    print(
        "Exons:",
        sum(
            exon_counts.values()
        )
    )

    print(
        "Transcripts without exons:",
        len(missing_exons)
    )

    return {
        "gtf_rows": rows,
        "query_transcripts":
            len(transcripts),
        "query_exons":
            sum(exon_counts.values())
    }

In [8]:
gtf_qc = inspect_gtf(
    PREPARED_GTF
)

GTF rows: 914843
Transcripts: 102140
Exons: 812703
Transcripts without exons: 0


In [10]:
prov = pd.read_csv(
    INDEX_PROVENANCE,
    sep="\t"
)

INDEX_DIR = Path(
    prov.iloc[0]["index_root"]
)

print(
    "Using index:",
    INDEX_DIR
)

Using index: Data/official_57_sample_index/extracted/chimpanzee_index


In [11]:
meta_file = (
    INDEX_DIR / "meta.txt"
)

lines = [
    line.strip()
    for line in meta_file.read_text().splitlines()
    if line.strip()
]

In [12]:
# Handle simple one-column or tabular meta.txt

if "\t" not in lines[0]:

    # First line may be a header
    sample_names = lines[1:]

else:

    if (
        lines[0]
        .split("\t")[0]
        .lower()
        == "name"
    ):

        sample_names = [
            x.split("\t")[0]
            for x in lines[1:]
        ]

    else:

        sample_names = [
            x.split("\t")[0]
            for x in lines
        ]

In [13]:
print(
    "Samples in official index:",
    len(sample_names)
)

print(
    sample_names[:10]
)

Samples in official index: 59
['SRR17660891', 'SRR17660893', 'SRR17660908', 'SRR17660909', 'SRR17660910', 'SRR17660911', 'SRR17660912', 'SRR17660913', 'SRR17660914', 'SRR17660915']


In [14]:
requested_output = (
    OUT_DIR / "annotated.gtf"
)

In [15]:
command = [
    ISOPEDIA,
    "isoform",

    "-g",
    str(PREPARED_GTF),

    "-i",
    str(INDEX_DIR),

    "-o",
    str(requested_output),

    "-n",
    str(THREADS),

    "--info"
]

print(
    "Running:",
    " ".join(command)
)

Running: Data/isopedia isoform -g Data/group1/Ensembl_reference_transcripts.sorted.gtf -i Data/official_57_sample_index/extracted/chimpanzee_index -o Output_files/official_57_sample/annotated.gtf -n 16 --info


In [17]:
subprocess.run(
    command,
    check=True
)

Isopedia version: 1.6.6


Parsed arguments:
{
  "idxdir": "Data/official_57_sample_index/extracted/chimpanzee_index",
  "gtf": "Data/group1/Ensembl_reference_transcripts.sorted.gtf",
  "flank": 10,
  "min_read": 1,
  "output": "Output_files/official_57_sample/annotated.gtf",
  "info": true,
  "num_threads": 16,
  "em_max_iter": 100,
  "em_conv_min_diff": 0.01,
  "em_chunk_size": 4,
  "em_effective_len_coef": 2,
  "em_damping_factor": 0.3,
  "min_em_abundance": 0.0001,
  "no_check_tss_tes": false,
  "tss_degrad_bp": 2000,
  "tes_degrad_bp": 8000,
  "terminal_tolerance_bp": 10,
  "cached_nodes": 10,
  "cached_chunk_num": 4,
  "cached_chunk_size_mb": 128,
  "verbose": false,
  "output_tmp_shard_counts": 10000
}
[2026-08-27T15:37:49Z INFO  isopedia::cmd::isoform] Loading GTF file...
[2026-08-27T15:37:50Z INFO  isopedia::cmd::isoform] Loaded 102,140 transcripts from gtf file
[2026-08-27T15:37:50Z INFO  isopedia::cmd::isoform] Loading index file
[2026-08-27T15:37:50Z INFO  isopedia::meta] If tab is detected in the li

CompletedProcess(args=['Data/isopedia', 'isoform', '-g', 'Data/group1/Ensembl_reference_transcripts.sorted.gtf', '-i', 'Data/official_57_sample_index/extracted/chimpanzee_index', '-o', 'Output_files/official_57_sample/annotated.gtf', '-n', '16', '--info'], returncode=0)

In [18]:
print(
    "Output exists:",
    ANNOTATED_OUTPUT.exists()
)

Output exists: True


In [19]:
with gzip.open(
    ANNOTATED_OUTPUT,
    "rt"
) as f:

    for i, line in enumerate(f):

        print(
            line.rstrip()
        )

        if i == 15:
            break

##[SAMPLE]Sample
##[SAMPLE]SRR17660891
##[SAMPLE]SRR17660893
##[SAMPLE]SRR17660908
##[SAMPLE]SRR17660909
##[SAMPLE]SRR17660910
##[SAMPLE]SRR17660911
##[SAMPLE]SRR17660912
##[SAMPLE]SRR17660913
##[SAMPLE]SRR17660914
##[SAMPLE]SRR17660915
##[SAMPLE]SRR22306524
##[SAMPLE]SRR22838393
##[SAMPLE]SRR22838394
##[SAMPLE]SRR22838395
##[SAMPLE]SRR22838396


In [20]:
def count_results(path):

    header_found = False

    rows = 0

    with gzip.open(
        path,
        "rt"
    ) as f:

        for line in f:

            if line.startswith(
                "#chrom\t"
            ):

                header_found = True
                continue

            if (
                header_found
                and line.strip()
            ):

                rows += 1

    return rows

In [21]:
returned_transcripts = (
    count_results(
        ANNOTATED_OUTPUT
    )
)

unreturned = (
    gtf_qc["query_transcripts"]
    - returned_transcripts
)

print(
    "Queried transcripts:",
    gtf_qc["query_transcripts"]
)

print(
    "Returned transcripts:",
    returned_transcripts
)

print(
    "Not returned:",
    unreturned
)

Queried transcripts: 102140
Returned transcripts: 102140
Not returned: 0


In [22]:
annotation_provenance = pd.DataFrame([
    {
        "group1_gtf":
            str(GTF.resolve()),

        "prepared_gtf":
            str(PREPARED_GTF.resolve()),

        "query_transcripts":
            gtf_qc[
                "query_transcripts"
            ],

        "query_exons":
            gtf_qc[
                "query_exons"
            ],

        "index_root":
            str(INDEX_DIR.resolve()),

        "index_samples":
            len(sample_names),

        "annotated_output":
            str(ANNOTATED_OUTPUT),

        "returned_transcripts":
            returned_transcripts,

        "unreturned_transcripts":
            unreturned
    }
])

display(
    annotation_provenance
)

,group1_gtf,prepared_gtf,query_transcripts,query_exons,index_root,index_samples,annotated_output,returned_transcripts,unreturned_transcripts
0,/home/nilabjab/Assignment/hackathon_26/Scripts...,/home/nilabjab/Assignment/hackathon_26/Scripts...,102140,812703,/home/nilabjab/Assignment/hackathon_26/Scripts...,59,Output_files/official_57_sample/annotated.gtf.gz,102140,0


In [23]:
annotation_provenance.to_csv(
    OUT_DIR
    / "annotation_provenance.tsv",
    sep="\t",
    index=False
)